# Análise e Limpeza de Dados de Tipo de Veículo em Acidentes de Trânsito

Este notebook apresenta o processo de análise, limpeza e preparação dos dados de tipos de veículos envolvidos em acidentes de trânsito do município de São Paulo, utilizando a base RENAEST. O objetivo é preparar os dados para análises estatísticas e geração de insights relevantes para a segurança viária.

## Instalação e Importação de Bibliotecas
Certifique-se de que o pandas está instalado. Em seguida, importe as bibliotecas necessárias.

In [ ]:
%pip install pandas

In [3]:
import pandas as pd
import os
import matplotlib.pyplot as plt

## Carregamento dos Dados
Carregue o arquivo CSV com os dados de tipos de veículos. O caminho deve ser ajustado conforme a estrutura de pastas do projeto.

In [4]:
csv_path = 'dataset/renaest_dabertos_20250412/TipoVeiculo_DadosAbertos_20250412.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path, sep=';', encoding='utf-8', low_memory=False)
    print('Arquivo carregado com sucesso!')
else:
    print(f"Arquivo não encontrado: {csv_path}")

Arquivo carregado com sucesso!


## Inspeção Inicial dos Dados
Visualize as primeiras linhas, informações gerais e estatísticas descritivas do dataset.

In [5]:
display(df.head())
df.info()
df.describe()

,num_acidente,tipo_veiculo,ind_veic_estrangeiro,qtde_veiculos
0,1,NAO INFORMADO,NAO INFORMADO,1
1,119,NAO INFORMADO,NAO INFORMADO,2
2,121,NAO INFORMADO,NAO INFORMADO,1
3,141,NAO INFORMADO,NAO INFORMADO,1
4,142,NAO INFORMADO,NAO INFORMADO,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8296666 entries, 0 to 8296665
Data columns (total 4 columns):
 #   Column                Dtype 
---  ------                ----- 
 0   num_acidente          int64 
 1   tipo_veiculo          object
 2   ind_veic_estrangeiro  object
 3   qtde_veiculos         int64 
dtypes: int64(2), object(2)
memory usage: 253.2+ MB


,num_acidente,qtde_veiculos
count,8.296666e+06,8.296666e+06
mean,3.519253e+06,1.187476e+00
std,2.032334e+06,4.326238e-01
min,1.000000e+00,1.000000e+00
25%,1.759197e+06,1.000000e+00
50%,3.519212e+06,1.000000e+00
75%,5.279008e+06,1.000000e+00
max,7.039437e+06,3.800000e+01


## Limpeza Inicial dos Dados
Remova colunas totalmente vazias e linhas duplicadas.

In [6]:
df_limpo = df.dropna(axis=1, how='all')
df_limpo = df_limpo.drop_duplicates()
display(df_limpo.head())
df_limpo.info()

,num_acidente,tipo_veiculo,ind_veic_estrangeiro,qtde_veiculos
0,1,NAO INFORMADO,NAO INFORMADO,1
1,119,NAO INFORMADO,NAO INFORMADO,2
2,121,NAO INFORMADO,NAO INFORMADO,1
3,141,NAO INFORMADO,NAO INFORMADO,1
4,142,NAO INFORMADO,NAO INFORMADO,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8296666 entries, 0 to 8296665
Data columns (total 4 columns):
 #   Column                Dtype 
---  ------                ----- 
 0   num_acidente          int64 
 1   tipo_veiculo          object
 2   ind_veic_estrangeiro  object
 3   qtde_veiculos         int64 
dtypes: int64(2), object(2)
memory usage: 253.2+ MB


## Filtragem dos Dados para São Paulo a partir de 2020
Selecione apenas os registros de acidentes ocorridos a partir de 2020, no município de São Paulo.

In [ ]:
df_sp = df_limpo[(df_limpo['ano_acidente'] >= 2020) & (df_limpo['uf_acidente'] == 'SP') & (df_limpo['chv_localidade'].str.startswith('SP3550308'))]
display(df_sp.head())

### Contar valores únicos em todas as colunas
Esta função retorna a quantidade de valores únicos e os próprios valores únicos para cada coluna do DataFrame.

In [ ]:
def contar_valores_unicos_todas_colunas(df):
    resultado = {}
    for col in df.columns:
        n_unicos = df[col].nunique()
        valores_unicos = df[col].unique()
        resultado[col] = {'quantidade': n_unicos, 'valores': valores_unicos}
    return resultado

contagem_unicos = contar_valores_unicos_todas_colunas(df_sp)
for coluna, info in contagem_unicos.items():
    print(f'{coluna}: {info["quantidade"]} valores únicos')

### Função para contar valores 'DESCONHECIDO' ou 'NAO INFORMADO' em todas as colunas
Esta função percorre todas as colunas do DataFrame e retorna a contagem desses valores para cada coluna.

In [ ]:
def contar_desconhecido_ou_nao_informado(df):
    resultado = {}
    for col in df.columns:
        count = df[col].isin(['DESCONHECIDO', 'NAO INFORMADO']).sum()
        if count > 0:
            resultado[col] = count
    return resultado

contagem = contar_desconhecido_ou_nao_informado(df_sp)
print('Contagem de "DESCONHECIDO" ou "NAO INFORMADO" por coluna:')
for coluna, qtd in contagem.items():
    print(f'{coluna}: {qtd}')

## Remoção de Colunas Irrelevantes
Removeremos do dataset as colunas que não são necessárias para a análise ou que possuem apenas valores ausentes ou redundantes. Ajuste a lista conforme a análise anterior.

In [ ]:
colunas_remover = [
    'chv_localidade', 'latitude_acidente', 'longitude_acidente', 
    'uf_acidente', 'ano_acidente', 'mes_acidente', 'mes_ano_acidente',
    'bairro_acidente', 'cep_acidente'
]
df_sp_limpo = df_sp.drop(columns=[col for col in colunas_remover if col in df_sp.columns])
display(df_sp_limpo.head())

## Salvar Dataset Limpo
Salve o DataFrame limpo em um novo arquivo CSV para uso posterior.

In [ ]:
output_path = 'dataset/renaest_dabertos_20250412/TipoVeiculo_DadosAbertos_20250412_limpo.csv'
df_sp_limpo.to_csv(output_path, sep=';', index=False, encoding='utf-8')
print(f"Dataset limpo salvo em: {output_path}")

## Considerações Finais
O dataset foi limpo, filtrado e salvo, estando pronto para análises estatísticas, geração de gráficos e elaboração de relatórios acadêmicos.

## Sugestões de Análises Estatísticas dos Tipos de Veículo

- Distribuição dos tipos de veículos envolvidos em acidentes
- Veículos mais frequentes em acidentes fatais
- Relação entre tipo de veículo e gravidade do acidente
- Veículos por horário e dia da semana

Adapte e expanda conforme o foco do seu trabalho acadêmico.

In [7]:
# Exemplo: Distribuição dos tipos de veículos
if 'tipo_veiculo' in df_sp_limpo.columns:
    tipos = df_sp_limpo['tipo_veiculo'].value_counts()
    tipos.plot(kind='bar', figsize=(10,5))
    plt.title('Distribuição dos Tipos de Veículo Envolvidos em Acidentes')
    plt.xlabel('Tipo de Veículo')
    plt.ylabel('Quantidade de Registros')
    plt.tight_layout()
    plt.show()
else:
    print('Coluna tipo_veiculo não encontrada no dataset.')

NameError: name 'df_sp_limpo' is not defined